# Tutorial: HSIC-ANOVA for Hierarchical Design Spaces (ADSG)

This tutorial demonstrates how to use the `HsicAnovaAdsg` explainer to compute global sensitivity indices on a hierarchical design space. We will build a simple conditional problem where a root variable dictates the activation of a decreed variable, causing structural sparsity.

In [ ]:
import numpy as np
from smt.utils.design_space import DesignSpace, FloatVariable, IntegerVariable
from smt.surrogate_models import KRG
from smt_explainability.hsic_anova import HsicAnovaAdsg
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Defining the Hierarchical Design Space

Let's create a space with 2 variables:
- `x1`: Root variable in [0, 1].
- `x2`: Decreed variable in [0, 1], only active when `x1 > 0.5`.

In [ ]:
ds = DesignSpace([
    FloatVariable(0, 1),
    FloatVariable(0, 1)
])

# Declare x2 as acting only if x1 > 0.5
ds.declare_decreed_var(decreed_var=1, meta_var=0, meta_value=[0.5, 1.0])

## 2. Sampling and Mock Function

We generate samples and evaluate a mock function: `y = x1 + x2` (when `x2` is active).

In [ ]:
# Sample inputs
from smt.sampling_methods import LHS
sampler = LHS(xlimits=ds.get_num_bounds(), criterion='ese', random_state=42)
x_samp = sampler(200)

# Impute inactive x2 to a dummy value (e.g., 0.5)
_, is_acting = ds.correct_get_acting(x_samp)
x_samp[~is_acting[:, 1], 1] = 0.5

# Mock output
y_samp = x_samp[:, 0] + np.where(is_acting[:, 1], x_samp[:, 1], 0.0)

# Train a simple surrogate (Kriging)
sm = KRG(design_space=ds, print_global=False)
sm.set_training_values(x_samp, y_samp)
sm.train()

## 3. HSIC-ANOVA Analysis

We initialize `HsicAnovaAdsg` which will automatically parse the `DesignSpace` to handle the conditional distances correctly.

In [ ]:
explainer = HsicAnovaAdsg(model=sm, ds=ds)

# Compute HSIC-ANOVA decomposition
results, total_hsic = explainer.explain(X=x_samp, max_order=2, var_names=["x1", "x2"])

print(f"Total HSIC: {total_hsic:.4f}\n")
print("Order | Global Var % | Intrinsic Var % | Act. Freq | Variables")
print("-" * 65)
for res in results:
    glob_var = (res["trace"] / total_hsic) * 100
    int_var = (res["adj_trace"] / total_hsic) * 100
    p_a = res["p_A"] * 100
    print(f"{res['order']:^5} | {glob_var:^12.2f} | {int_var:^15.2f} | {p_a:^9.1f} | {res['name']}")